# Modul 06: Datenaufteilung, Verluste und Metriken

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Daten aufteilen, Verluste und Metriken  
    **Erwarteter Schwierigkeitsgrad:** Leicht fortgeschritten  
    **Orientierungszeit:** etwa 100 bis 135 Minuten

    ## Überblick

    Sie erstellen reproduzierbare zufällige, stratifizierte, gruppenbasierte und zeitliche Aufteilungen. Anschließend berechnen Sie Regressions- und Klassifikationsmetriken einschließlich Baselines, Schwellenwerten und Konfusionsmatrix.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_06A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_06B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes erstellen.
- Klassen, Gruppen und Zeitordnung bei Splits berücksichtigen.
- Typische Formen von Zielwert- und Vorverarbeitungsleckage erkennen.
- Regressionsverluste aus tatsächlichen Werten und Vorhersagen berechnen.
- Einfache Mittelwert-, Median- und Mehrheitsklassen-Baselines erstellen.
- Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen berechnen.

    ## Bewertete Fähigkeiten

    - Indexsplits, Stratifikation, GroupShuffleSplit und Zeitaufteilung
- disjunkte Mengen und Splitprotokolle prüfen
- MAE, MSE, RMSE und Baselines manuell berechnen
- Konfusionsmatrix, Precision, Recall, F1 und Log Loss verstehen
- Skalierung ausschließlich auf Trainingsdaten fitten

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_classification, y_classification = make_classification(
    n_samples=240,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    weights=[0.78, 0.22],
    class_sep=1.1,
    random_state=RANDOM_SEED,
)
classification_groups = np.repeat(np.arange(60), 4)
classification_data = pd.DataFrame(
    X_classification,
    columns=[f"feature_{i}" for i in range(X_classification.shape[1])],
)
classification_data["target"] = y_classification
classification_data["group_id"] = classification_groups

# Zeitlich geordnete Daten für eine kleine Prognoseaufgabe.
time_index = pd.date_range("2026-01-01", periods=180, freq="D")
time_feature = np.arange(180, dtype=float)
time_target = 20 + 0.08 * time_feature + 2.0 * np.sin(time_feature / 12) + rng.normal(0, 0.8, 180)
temporal_data = pd.DataFrame(
    {"date": time_index, "time_feature": time_feature, "target": time_target}
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Train-, Validierungs- und Testindizes manuell erzeugen

    Erzeugen Sie für 100 Beobachtungen reproduzierbare, zufällig gemischte Indizes mit 60 Prozent Training, 20 Prozent Validierung und 20 Prozent Test.

1. Verwenden Sie `np.random.default_rng(RANDOM_SEED)`.
2. Prüfen Sie die Größen der drei Mengen.
3. Prüfen Sie, dass keine Überschneidungen existieren.
4. Prüfen Sie, dass zusammen alle Indizes von 0 bis 99 genau einmal vorkommen.

> **Hinweis:** Prüfen Sie nicht nur die Längen, sondern auch Überschneidung und vollständige Abdeckung.

In [ ]:
n_samples = 100

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Train-, Validierungs- und Testindizes manuell erzeugen
#
# Ziel dieser Codezelle:
# Erzeugen Sie für 100 Beobachtungen reproduzierbare, zufällig gemischte Indizes mit
# 60 Prozent Training, 20 Prozent Validierung und 20 Prozent Test. 1. Verwenden Sie
# np.random.defaultrng(RANDOMSEED). 2. Prüfen Sie die...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

n_samples = 100

# Ein lokaler Generator mit festem Startwert macht die Reihenfolge
# reproduzierbar, ohne globale Zufallszustände zu verändern.
split_rng = np.random.default_rng(RANDOM_SEED)
shuffled_indices = split_rng.permutation(n_samples)

# Die Grenzen werden aus den gewünschten Anteilen berechnet.
train_end = int(0.60 * n_samples)
validation_end = int(0.80 * n_samples)

train_indices = shuffled_indices[:train_end]
validation_indices = shuffled_indices[train_end:validation_end]
test_indices = shuffled_indices[validation_end:]

# Mengenoperationen prüfen Überschneidungen unabhängig von Reihenfolge.
train_set = set(train_indices.tolist())
validation_set = set(validation_indices.tolist())
test_set = set(test_indices.tolist())

assert len(train_indices) == 60
assert len(validation_indices) == 20
assert len(test_indices) == 20
assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)
assert train_set | validation_set | test_set == set(range(n_samples))

print("Train:", len(train_indices), train_indices[:8])
print("Validierung:", len(validation_indices), validation_indices[:8])
print("Test:", len(test_indices), test_indices[:8])

### Reflexion zu Aufgabe 1

Die drei Mengen müssen disjunkt sein, weil eine Beobachtung nicht gleichzeitig zum Lernen und zur unabhängigen Bewertung gehören darf. Der Validierungssatz unterstützt Modellauswahl und Entscheidungen während der Entwicklung. Der Testsatz wird erst für die abschließende, möglichst unbeeinflusste Bewertung verwendet.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Stratifizierte, gruppenbasierte und zeitliche Splits vergleichen

    Erstellen Sie drei passende Aufteilungen.

1. Stratifizierter Train/Test-Split für `classification_data`, sodass die Klassenanteile ähnlich bleiben.
2. Gruppenbasierter Split, sodass keine `group_id` in Train und Test gleichzeitig vorkommt.
3. Zeitlicher Split von `temporal_data`, bei dem die ersten 75 Prozent Training und die letzten 25 Prozent Test bilden.

Erstellen Sie eine kleine Kontrolltabelle mit Größen, Klassenanteilen, Gruppenüberschneidung und Datumsbereichen.

> **Hinweis:** Wählen Sie den Split nach der Entstehungsstruktur der Beobachtungen, nicht nur nach Bequemlichkeit.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Stratifizierte, gruppenbasierte und zeitliche Splits vergleichen
#
# Ziel dieser Codezelle:
# Erstellen Sie drei passende Aufteilungen. 1. Stratifizierter Train/Test-Split für
# classificationdata, sodass die Klassenanteile ähnlich bleiben. 2. Gruppenbasierter
# Split, sodass keine groupid in Train und Test gleich...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

feature_columns = [c for c in classification_data.columns if c.startswith("feature_")]
X = classification_data[feature_columns]
y = classification_data["target"]

# Stratifikation hält den Anteil der seltenen Klasse in beiden Mengen
# ungefähr konstant.
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

# GroupShuffleSplit trennt vollständige Gruppen. Die Zielklassen werden
# hier nicht automatisch stratifiziert.
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_SEED,
)
train_group_idx, test_group_idx = next(
    group_splitter.split(
        X,
        y,
        groups=classification_data["group_id"],
    )
)
train_groups = set(classification_data.iloc[train_group_idx]["group_id"])
test_groups = set(classification_data.iloc[test_group_idx]["group_id"])

# Bei Zeitdaten wird nicht gemischt. Der Test simuliert zukünftige Daten.
temporal_sorted = temporal_data.sort_values("date").reset_index(drop=True)
temporal_cut = int(0.75 * len(temporal_sorted))
temporal_train = temporal_sorted.iloc[:temporal_cut]
temporal_test = temporal_sorted.iloc[temporal_cut:]

split_control = pd.DataFrame(
    [
        {
            "split": "stratifiziert",
            "train_size": len(y_train_s),
            "test_size": len(y_test_s),
            "train_positive_share": y_train_s.mean(),
            "test_positive_share": y_test_s.mean(),
            "group_overlap": np.nan,
            "train_range": "zufällig",
            "test_range": "zufällig",
        },
        {
            "split": "gruppenbasiert",
            "train_size": len(train_group_idx),
            "test_size": len(test_group_idx),
            "train_positive_share": y.iloc[train_group_idx].mean(),
            "test_positive_share": y.iloc[test_group_idx].mean(),
            "group_overlap": len(train_groups & test_groups),
            "train_range": "mehrere Gruppen",
            "test_range": "getrennte Gruppen",
        },
        {
            "split": "zeitlich",
            "train_size": len(temporal_train),
            "test_size": len(temporal_test),
            "train_positive_share": np.nan,
            "test_positive_share": np.nan,
            "group_overlap": np.nan,
            "train_range": f"{temporal_train['date'].min().date()} bis {temporal_train['date'].max().date()}",
            "test_range": f"{temporal_test['date'].min().date()} bis {temporal_test['date'].max().date()}",
        },
    ]
)

assert train_groups.isdisjoint(test_groups)
assert temporal_train["date"].max() < temporal_test["date"].min()

print(split_control.round(3).to_string(index=False))

### Reflexion zu Aufgabe 2

Stratifikation ist wichtig, wenn Klassenanteile erhalten bleiben sollen. Gruppenbasierte Aufteilung verhindert, dass eng verwandte Beobachtungen derselben Person, Maschine oder Quelle in beiden Mengen auftreten. Zeitliche Aufteilung simuliert eine Zukunftsprognose und darf die Reihenfolge nicht durchmischen. Kein einzelner Split erfüllt automatisch alle drei Anforderungen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Datenleckage erkennen und Skalierung korrekt fitten

    Verwenden Sie `classification_data`.

1. Erzeugen Sie absichtlich ein Leckage-Merkmal `target_plus_noise = target + kleine Zufallsabweichung`.
2. Vergleichen Sie die Testgenauigkeit einer logistischen Regression mit und ohne dieses Merkmal auf demselben stratifizierten Split.
3. Skalieren Sie die legitimen Merkmale korrekt: `fit` nur auf Training, `transform` auf Training und Test.
4. Geben Sie Mittelwert und Standardabweichung der skalierten Trainings- und Testdaten aus.
5. Entfernen Sie das Leckage-Merkmal dauerhaft aus der Modellpipeline.

> **Hinweis:** Fragen Sie für jedes Merkmal: Wäre dieser Wert genau im Moment der Vorhersage bereits bekannt?

In [ ]:
leakage_data = classification_data.copy()

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Datenleckage erkennen und Skalierung korrekt fitten
#
# Ziel dieser Codezelle:
# Verwenden Sie classificationdata. 1. Erzeugen Sie absichtlich ein Leckage-Merkmal
# targetplusnoise = target + kleine Zufallsabweichung. 2. Vergleichen Sie die
# Testgenauigkeit einer logistischen Regression mit und ohne...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

leakage_data = classification_data.copy()

# Dieses Merkmal verwendet den Zielwert direkt und wäre bei echter
# Vorhersage nicht verfügbar. Es dient ausschließlich zur Demonstration.
leakage_data["target_plus_noise"] = (
    leakage_data["target"]
    + rng.normal(0, 0.03, size=len(leakage_data))
)

legitimate_features = [c for c in leakage_data.columns if c.startswith("feature_")]
leaked_features = legitimate_features + ["target_plus_noise"]
y = leakage_data["target"]

# Der Split wird über gemeinsame Indizes erzeugt, damit beide Modelle
# exakt dieselben Trainings- und Testbeobachtungen sehen.
all_indices = np.arange(len(leakage_data))
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_train_legit = leakage_data.loc[train_idx, legitimate_features]
X_test_legit = leakage_data.loc[test_idx, legitimate_features]
X_train_leaked = leakage_data.loc[train_idx, leaked_features]
X_test_leaked = leakage_data.loc[test_idx, leaked_features]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# Der Skalierer lernt Mittelwerte und Standardabweichungen ausschließlich
# aus dem Trainingssatz. Testinformationen beeinflussen den Fit nicht.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_legit)
X_test_scaled = scaler.transform(X_test_legit)

legitimate_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
legitimate_model.fit(X_train_scaled, y_train)
legitimate_accuracy = accuracy_score(
    y_test,
    legitimate_model.predict(X_test_scaled),
)

# Das Leckage-Modell erhält absichtlich direkte Zielinformation. Die
# hohe Leistung ist deshalb nicht vertrauenswürdig.
leaked_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
leaked_model.fit(X_train_leaked, y_train)
leaked_accuracy = accuracy_score(
    y_test,
    leaked_model.predict(X_test_leaked),
)

print("Legitime Genauigkeit:", round(legitimate_accuracy, 3))
print("Genauigkeit mit Zielwertleckage:", round(leaked_accuracy, 3))
print("Train-Mittelwerte nach Skalierung:", np.round(X_train_scaled.mean(axis=0), 3))
print("Train-Standardabweichungen:", np.round(X_train_scaled.std(axis=0), 3))
print("Test-Mittelwerte:", np.round(X_test_scaled.mean(axis=0), 3))

### Reflexion zu Aufgabe 3

Das Leckage-Merkmal kann eine nahezu perfekte Bewertung erzeugen, obwohl das Modell im echten Einsatz scheitern würde. Eine gute Testzahl ist nur dann aussagekräftig, wenn alle verwendeten Merkmale zum Vorhersagezeitpunkt verfügbar sind. Dass die Testdaten nach Skalierung nicht exakt Mittelwert null besitzen, ist normal. Die Skalierungsparameter stammen bewusst nur aus dem Training.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Regressionsverluste und Baselines manuell berechnen

    Verwenden Sie die unten angegebenen Trainings- und Testzielwerte.

1. Erzeugen Sie eine Mittelwert- und eine Median-Baseline aus `y_train_reg`.
2. Berechnen Sie für beide Baselines sowie für `model_predictions` MAE, MSE und RMSE ausschließlich mit NumPy.
3. Erstellen Sie eine sortierte Vergleichstabelle.
4. Zeigen Sie die Residuen des besten Verfahrens in einem Punktdiagramm.

> **Hinweis:** Berechnen Sie Baselinekonstanten nie aus dem Testsatz.

In [ ]:
y_train_reg = np.array([12, 14, 15, 15, 16, 18, 20, 22, 40], dtype=float)
y_test_reg = np.array([13, 16, 19, 24, 35], dtype=float)
model_predictions = np.array([14, 15, 20, 23, 30], dtype=float)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Regressionsverluste und Baselines manuell berechnen
#
# Ziel dieser Codezelle:
# Verwenden Sie die unten angegebenen Trainings- und Testzielwerte. 1. Erzeugen Sie
# eine Mittelwert- und eine Median-Baseline aus ytrainreg. 2. Berechnen Sie für
# beide Baselines sowie für modelpredictions MAE, MSE und R...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

y_train_reg = np.array([12, 14, 15, 15, 16, 18, 20, 22, 40], dtype=float)
y_test_reg = np.array([13, 16, 19, 24, 35], dtype=float)
model_predictions = np.array([14, 15, 20, 23, 30], dtype=float)

# Baselines dürfen nur Informationen aus dem Trainingssatz verwenden.
mean_value = float(np.mean(y_train_reg))
median_value = float(np.median(y_train_reg))
mean_predictions = np.full_like(y_test_reg, mean_value)
median_predictions = np.full_like(y_test_reg, median_value)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    # Residuen werden als tatsächlicher minus vorhergesagter Wert definiert.
    residuals = y_true - y_pred
    mae = float(np.mean(np.abs(residuals)))
    mse = float(np.mean(residuals**2))
    rmse = float(np.sqrt(mse))
    return {"MAE": mae, "MSE": mse, "RMSE": rmse}

prediction_sets = {
    "Mittelwert-Baseline": mean_predictions,
    "Median-Baseline": median_predictions,
    "Beispielmodell": model_predictions,
}

comparison_rows = []
for name, predictions in prediction_sets.items():
    metrics = regression_metrics(y_test_reg, predictions)
    comparison_rows.append({"method": name, **metrics})

regression_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

best_method = regression_comparison.loc[0, "method"]
best_predictions = prediction_sets[best_method]
best_residuals = y_test_reg - best_predictions

print(regression_comparison.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(best_predictions, best_residuals, s=70)
ax.axhline(0, linewidth=1)
ax.set_title(f"Residuen: {best_method}")
ax.set_xlabel("Vorhersage")
ax.set_ylabel("Tatsächlich minus Vorhersage")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

MAE gewichtet alle absoluten Fehler linear. MSE und RMSE bestrafen große Fehler stärker, weil Residuen quadriert werden. Der Trainingsmittelwert wird durch den Wert 40 nach oben gezogen, während der Median robuster ist. Eine Baseline ist kein Endmodell, sondern ein Mindestvergleich: Ein komplexeres Verfahren sollte sie unter der gewählten Metrik sinnvoll übertreffen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Schwellenwerte und Klassifikationsmetriken

    Gegeben sind wahre Labels und vorhergesagte Wahrscheinlichkeiten.

1. Erzeugen Sie Labels für Schwellenwerte 0.50 und 0.35.
2. Berechnen Sie TP, TN, FP und FN manuell.
3. Berechnen Sie Accuracy, Precision, Recall und F1 mit sicherer Behandlung möglicher Division durch null.
4. Berechnen Sie den probabilistischen Log Loss mit `sklearn.metrics.log_loss`.
5. Vergleichen Sie beide Schwellenwerte und empfehlen Sie einen für ein Szenario, in dem übersehene positive Fälle besonders teuer sind.

> **Hinweis:** Ein Schwellenwert ist eine Einsatzentscheidung und nicht zwingend ein fester Bestandteil des trainierten Modells.

In [ ]:
y_true = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0])
positive_scores = np.array([0.08, 0.30, 0.42, 0.18, 0.77, 0.61, 0.48, 0.52, 0.12, 0.88, 0.36, 0.25])

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Schwellenwerte und Klassifikationsmetriken
#
# Ziel dieser Codezelle:
# Gegeben sind wahre Labels und vorhergesagte Wahrscheinlichkeiten. 1. Erzeugen Sie
# Labels für Schwellenwerte 0.50 und 0.35. 2. Berechnen Sie TP, TN, FP und FN
# manuell. 3. Berechnen Sie Accuracy, Precision, Recall und F...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

y_true = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0])
positive_scores = np.array([0.08, 0.30, 0.42, 0.18, 0.77, 0.61, 0.48, 0.52, 0.12, 0.88, 0.36, 0.25])

def classification_metrics_at_threshold(
    y_true_values: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> dict:
    # Wahrscheinlichkeiten werden erst durch den Schwellenwert zu Labels.
    predictions = (scores >= threshold).astype(int)

    tp = int(np.sum((y_true_values == 1) & (predictions == 1)))
    tn = int(np.sum((y_true_values == 0) & (predictions == 0)))
    fp = int(np.sum((y_true_values == 0) & (predictions == 1)))
    fn = int(np.sum((y_true_values == 1) & (predictions == 0)))

    accuracy = (tp + tn) / len(y_true_values)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    return {
        "threshold": threshold,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

threshold_results = pd.DataFrame(
    [
        classification_metrics_at_threshold(
            y_true,
            positive_scores,
            threshold,
        )
        for threshold in [0.50, 0.35]
    ]
)

# Log Loss nutzt die Wahrscheinlichkeiten direkt und ist unabhängig
# vom später gewählten Klassifikationsschwellenwert.
probability_log_loss = log_loss(y_true, positive_scores)

print(threshold_results.round(3).to_string(index=False))
print("Log Loss der Wahrscheinlichkeiten:", round(probability_log_loss, 4))

### Reflexion zu Aufgabe 5

Der niedrigere Schwellenwert 0,35 erhöht typischerweise den Recall, weil mehr Fälle als positiv markiert werden. Dafür können mehr falsch-positive Fälle entstehen und Precision oder Accuracy sinken. Wenn übersehene positive Fälle besonders teuer sind, ist der niedrigere Schwellenwert häufig geeigneter, muss aber anhand konkreter Kosten und Kapazitäten gewählt werden. Log Loss bewertet zusätzlich, wie gut die Wahrscheinlichkeiten selbst zu den Labels passen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.